# 02c -- fact_rookie_rankings schema seed

**Purpose**: One-time schema initialization for `fact_rookie_rankings`. Wipes and
re-creates the empty parquet with correct dtypes.

> **WARNING**: Re-running this cell destroys all ingested rankings. Run only at
> season start or when the schema changes.

**Run sequence**:
1. This notebook — reset schema
2. `03a_fantasypros_rankings.ipynb`
3. `03b_walterfootball_rankings.ipynb`
4. `03c_ktc_rankings.ipynb`
5. `09_apply_fuzzy_review.ipynb` (if review file generated)

In [14]:
# ---- Setup & Config (shared module) ----------------------------------------
# All config + path anchoring + helpers live in notebooks/etl_helpers.py.
# CFG.data_dir / DATA / REVIEW are anchored to the repo root -> writes always
# land in <repo>/data no matter the CWD this notebook runs from.
import sys
from pathlib import Path
for _p in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import (LeagueConfig, CFG, DATA, REVIEW, TODAY, ALIAS,
                         clean_player_name, generate_player_key, parse_height_to_inches)

import pandas as pd
from pathlib import Path
from dataclasses import dataclass


CFG  = LeagueConfig()


In [15]:
# -- fact_rookie_rankings schema seed -----------------------------------------------
# Seed once; subsequent cells append rows. Re-running this cell resets the table.
# gsis_id      : primary FK -> dim_nfl_players (null pre-signing)
# player_key   : interim FK -> dim_rookie_prospect (null for non-current-class)
# source_name  : expert source key (e.g., FantasyPros_PPR, WalterFootball_QB, composite)
# source_site  : site group (FantasyPros | WalterFootball | composite)
# Composite dedup key: player_key + source_name + phase + draft_year
fact_rookie_rankings = pd.DataFrame({
    "player_key":      pd.Series(dtype="str"),    # interim FK -> dim_rookie_prospect
    "gsis_id":         pd.Series(dtype="str"),    # FK -> dim_nfl_players (null pre-signing)
    "source_name":     pd.Series(dtype="str"),    # expert source key
    "source_site":     pd.Series(dtype="str"),    # site group (FantasyPros | WalterFootball | composite)
    "phase":           pd.Series(dtype="str"),    # pre_combine | post_combine | post_draft | composite
    "draft_year":      pd.Series(dtype="Int64"),
    "global_rank":     pd.Series(dtype="Int64"),  # overall big-board rank
    "positional_rank": pd.Series(dtype="Int64"),  # rank within position_group
    "grade":           pd.Series(dtype="float64"),# 0-100 grade if provided by source
    "capture_date":    pd.Series(dtype="str"),    # ISO date the scraper ran
    "rank_date":       pd.Series(dtype="str"),    # ISO date the source last updated the ranking (null if unparseable)
})

out_path = DATA / "fact_rookie_rankings.parquet"
fact_rookie_rankings.to_parquet(out_path, index=False)
print(f"fact_rookie_rankings: schema seed written -> {out_path}")
print(fact_rookie_rankings.dtypes.to_string())

fact_rookie_rankings: schema seed written -> data\fact_rookie_rankings.parquet
player_key             str
gsis_id                str
source_name            str
source_site            str
phase                  str
draft_year           Int64
global_rank          Int64
positional_rank      Int64
grade              float64
capture_date           str
rank_date              str
